# STEW mental workload

This notebook covers:

- loading the STEW dataset  
- mapping labels to four classes: Baseline (BL), Low (LW), Medium (MW), High (HW)  
- preprocessing EEG and building subject-wise manifests  
- generating band-wise topographical image sequences (2 s frames, 10 s context)  
- training the VAE-CBAM-BLSTM model with subject-wise LOSO cross-validation  
- computing performance metrics and saving per-subject results  
- running baselines, ablations, and sensitivity analyses  
- computing Grad-CAM maps and summarizing frontal-parietal relevance


### Paths and imports

In [1]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Source path:", SRC)

/Users/osama/Downloads/Work/PhD/Mental Workload Prediction - Daily/Papers/Paper2/code


### Core libraries and project modules

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from stew_mwl.config import Config, CLASS_NAMES
from stew_mwl.data import (
    set_global_determinism,
    build_subject_manifest,
    read_signal_txt,
    preprocess_signal,
)
from stew_mwl.features import build_sequence_images
from stew_mwl.models import (
    build_vae,
    build_classifier_from_encoder,
    compile_classifier,
    run_loso_training,
    run_ablation_variants,
    run_baseline_models,
)
from stew_mwl.eval import (
    summarize_metrics,
    aggregate_fold_metrics,
    paired_ttest_vs_full,
    classification_tables,
)
from stew_mwl.gradcam import (
    make_gradcam_heatmap,
    summarize_gradcam_frontal_parietal,
)

Notes:

Ensure `Config` in `stew_mwl/config.py` defines:

- `seed`
- `latent_dim = 128`
- `window_sec = 10`
- `frame_sec = 2`
- `seq_len = 10`
- `cbam_reduction = 8`
- `cbam_kernel = 7`
- `blstm_units = 20`
- `n_folds = 48`
- `output_root`, `data_root`, etc.

### Configure experiment

In [ ]:
cfg = Config()
set_global_determinism(cfg.seed)

cfg.output_root.mkdir(parents=True, exist_ok=True)
cfg.osf_export_root = cfg.output_root / "osf_export"
cfg.osf_export_root.mkdir(parents=True, exist_ok=True)

print(cfg)
print("OSF export path:", cfg.osf_export_root)

## Dataset setup

1. Download the STEW dataset from the official source (IEEE DataPort, DOI: 10.21227/44r8-ya50) and extract it under `data/STEW/`.
2. Ensure the directory contains text files such as `sub01_lo.txt`, `sub01_hi.txt`, and the ratings file.
3. The code below constructs a subject-wise manifest for all 48 participants.

Baseline (resting) recordings form the BL class. Task recordings are grouped into LW, MW, and HW based on the 1-9 subjective ratings:

- 1-3 → Low Workload (LW)  
- 4-6 → Medium Workload (MW)  
- 7-9 → High Workload (HW)

This yields four workload levels: BL, LW, MW, HW.

### Build manifest

In [ ]:
manifest = build_subject_manifest(cfg)
display(manifest.head())
manifest.shape

### Class distribution

In [ ]:
manifest["task_label"].value_counts().sort_index()

### Build topographical sequences

The manuscript describes:

- EEG preprocessing with filtering, artifact mitigation, and re-referencing  
- 2-second localized segments as inputs to the VAE for topographical feature learning  
- 10-second windows to provide temporal context for classification  

The implementation below follows this logic by creating 10-second parent windows and, within each window, extracting a sequence of overlapping 2-second band-wise topographical frames. Each trial is thus represented as a sequence of 10 frames of size 80×60×3 (height × width × RGB channels).

### Generate sequence data

In [ ]:
X_seq, y_seq, subj_ids = build_sequence_images(cfg, manifest)
X_seq.shape, y_seq.shape, subj_ids.shape

# Expected: `(n_trials, seq_len, 80, 60, 3)`.

### Build VAE and classifier

In [ ]:
input_shape = (cfg.seq_len, 80, 60, 3)

vae = build_vae(input_shape=input_shape, latent_dim=cfg.latent_dim)

classifier = build_classifier_from_encoder(
    vae.encoder,
    cbam_reduction=cfg.cbam_reduction,
    cbam_kernel=cfg.cbam_kernel,
    blstm_units=cfg.blstm_units,
    n_classes=len(CLASS_NAMES),
)

compile_classifier(classifier, cfg)

classifier.summary()


# this to Ensure `build_classifier_from_encoder`:
# - Applies CBAM (channel → spatial) with reduction 8 and 7×7 kernel.
# - Uses a time-distributed 2D CNN + flatten + BLSTM(20) + batch norm + dense(4, softmax).

### LOSO training for full model

In [ ]:
full_results_df, loso_splits = run_loso_training(
    cfg,
    classifier,
    vae,
    X_seq,
    y_seq,
    subj_ids,
)

full_results_df.to_csv(
    cfg.osf_export_root / "full_model_loso_metrics.csv",
    index=False,
)

aggregate_fold_metrics(full_results_df)


`run_loso_training` should:

- Loop over 48 subjects (LOSO).
- Train VAE + classifier per fold.
- Return a DataFrame with columns like `subject`, `accuracy`, `macro_f1`, `balanced_accuracy`, `kappa`.


### Classification tables

In [ ]:
classification_tables(full_results_df, CLASS_NAMES)

### Baseline models (PSD-SVM, CNN, BLSTM-LSTM)

In [ ]:
baseline_results = run_baseline_models(
    cfg,
    manifest,
    loso_splits,
)

for name, df in baseline_results.items():
    out_path = cfg.osf_export_root / f"baseline_{name}_metrics.csv"
    df.to_csv(out_path, index=False)
    print(name, "aggregate:")
    aggregate_fold_metrics(df)

`run_baseline_models` should return a dict:

- `{"psd_svm": df_psd, "cnn": df_cnn, "blstm_lstm": df_blstm_lstm}`.

### Statistical comparison vs strongest baseline

In [ ]:
blstm_lstm_df = baseline_results["blstm_lstm"]

p_val = paired_ttest_vs_full(full_results_df, blstm_lstm_df, metric="accuracy")
print("Full vs BLSTM-LSTM, p-value (accuracy):", p_val)

### Ablation variants (no VAE, no CBAM, uni-LSTM, CNN-only)

In [ ]:
ablation_results = run_ablation_variants(
    cfg,
    X_seq,
    y_seq,
    subj_ids,
    loso_splits,
)

for name, df in ablation_results.items():
    out_path = cfg.osf_export_root / f"ablation_{name}_metrics.csv"
    df.to_csv(out_path, index=False)
    print(name, "aggregate:")
    aggregate_fold_metrics(df)

### Statistical tests for ablations

In [ ]:
for name, df in ablation_results.items():
    p_val = paired_ttest_vs_full(full_results_df, df, metric="accuracy")
    print(f"Full vs {name}: p-value (accuracy) = {p_val:.4g}")

These p-values correspond to the ablation-statistics sentences in the Results.

### Sensitivity analyses: latent size, window, BLSTM steps, CBAM config

In [ ]:
latent_sweep_df = pd.read_csv(cfg.output_root / "sensitivity_latent_dim_internal.csv")
window_sweep_df = pd.read_csv(cfg.output_root / "sensitivity_window_internal.csv")
blstm_sweep_df = pd.read_csv(cfg.output_root / "sensitivity_blstm_internal.csv")
cbam_sweep_df = pd.read_csv(cfg.output_root / "sensitivity_cbam_internal.csv")

latent_sweep_df.to_csv(cfg.osf_export_root / "sensitivity_latent_dim.csv", index=False)
window_sweep_df.to_csv(cfg.osf_export_root / "sensitivity_window_length.csv", index=False)
blstm_sweep_df.to_csv(cfg.osf_export_root / "sensitivity_blstm_steps.csv", index=False)
cbam_sweep_df.to_csv(cfg.osf_export_root / "sensitivity_cbam_config.csv", index=False)

latent_sweep_df

### Grad-CAM example and summary

In [ ]:
# Example: compute Grad-CAM for a batch of test samples from the last LOSO fold
# Assumes you kept the trained classifier and some X_test, y_test for the last fold.

# gradcam_maps: shape (n_samples, seq_len, 80, 60) or similar
gradcam_maps, gradcam_meta = make_gradcam_heatmap(
    cfg,
    classifier,
    X_seq,
    y_seq,
    subj_ids,
    loso_splits,
)

# Summarize frontal/parietal importance per class
gradcam_summary_df = summarize_gradcam_frontal_parietal(
    cfg,
    gradcam_maps,
    gradcam_meta,
)

gradcam_summary_df.to_csv(
    cfg.osf_export_root / "gradcam_frontal_parietal_summary.csv",
    index=False,
)

gradcam_summary_df

### OSF export and reproducibility

All per-subject LOSO metrics for the full model, baseline models, and ablation variants, together with sensitivity-analysis summaries and Grad-CAM frontal-parietal importance scores, are saved under `output/osf_export/`.

These files correspond to the derived data deposited in the OSF repository cited in the manuscript and satisfy the PLOS ONE data and code sharing requirements.

## Build sequences

The manuscript describes:
- preprocessing with filtering and artifact handling
- 2-second localized inputs for topographical feature learning
- 10-second temporal context for classification

The implementation below follows that logic by creating 10-second parent windows and then extracting a sequence of overlapping 2-second topographical frames from each parent window.

In [ ]:
def build_dataset_for_subjects(subject_rows, cfg, window_seconds=None, sequence_length=None):
    X, y, subjects = [], [], []
    for _, row in subject_rows.iterrows():
        subject = int(row["subject"])

        baseline = preprocess_signal(read_signal_txt(Path(row["lo_path"])), cfg)
        task = preprocess_signal(read_signal_txt(Path(row["hi_path"])), cfg)

        x_bl = build_sequence_images(baseline, cfg, window_seconds=window_seconds, sequence_length=sequence_length)
        x_task = build_sequence_images(task, cfg, window_seconds=window_seconds, sequence_length=sequence_length)

        if len(x_bl):
            X.append(x_bl)
            y.append(np.full(len(x_bl), cfg.class_to_id["baseline"], dtype=np.int64))
            subjects.append(np.full(len(x_bl), subject, dtype=np.int64))

        if len(x_task):
            X.append(x_task)
            y.append(np.full(len(x_task), cfg.class_to_id[row["task_label"]], dtype=np.int64))
            subjects.append(np.full(len(x_task), subject, dtype=np.int64))

    X = np.concatenate(X, axis=0)
    y = np.concatenate(y, axis=0)
    subjects = np.concatenate(subjects, axis=0)
    return X, y, subjects

In [ ]:
# quick sanity build on a small subset
sample_rows = manifest.head(3)
X_sample, y_sample, s_sample = build_dataset_for_subjects(sample_rows, cfg)
X_sample.shape, y_sample.shape, np.unique(y_sample, return_counts=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for i in range(3):
    axes[i].imshow(X_sample[0, i])
    axes[i].set_title(f"Frame {i+1}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## Model helpers

In [ ]:
def train_vae_on_frames(x_train, cfg):
    frames = x_train.reshape(-1, cfg.image_h, cfg.image_w, x_train.shape[-1]).astype("float32")
    vae, encoder, decoder = build_vae(
        image_shape=(cfg.image_h, cfg.image_w, x_train.shape[-1]),
        latent_dim=cfg.latent_dim,
    )
    vae.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate))
    hist = vae.fit(
        frames,
        epochs=cfg.vae_epochs,
        batch_size=cfg.batch_size,
        shuffle=True,
        verbose=0,
    )
    return vae, encoder, decoder, hist

def train_classifier(x_train, y_train, x_val, y_val, cfg, use_cbam=True, use_encoder=True, bidirectional=True, sequence_model="lstm"):
    model = build_classifier_from_encoder(
        frame_shape=(cfg.image_h, cfg.image_w, x_train.shape[-1]),
        sequence_length=x_train.shape[1],
        latent_dim=cfg.latent_dim,
        n_classes=len(CLASS_NAMES),
        dropout=cfg.dropout,
        use_cbam=use_cbam,
        use_encoder=use_encoder,
        bidirectional=bidirectional,
        sequence_model=sequence_model,
    )
    model = compile_classifier(model, cfg.learning_rate)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    ]
    hist = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=cfg.clf_epochs,
        batch_size=cfg.batch_size,
        verbose=0,
        callbacks=callbacks,
    )
    return model, hist

## Full LOSO experiment

In [ ]:
def run_loso_full_model(cfg):
    rows = []
    fold_preds = []
    subjects = manifest["subject"].tolist()
    if cfg.loso_subjects_limit is not None:
        subjects = subjects[:cfg.loso_subjects_limit]

    for subject in subjects:
        train_rows = manifest[manifest["subject"] != subject]
        test_rows = manifest[manifest["subject"] == subject]

        x_train, y_train, _ = build_dataset_for_subjects(train_rows, cfg)
        x_test, y_test, _ = build_dataset_for_subjects(test_rows, cfg)

        # simple validation split from train
        rng = np.random.default_rng(cfg.seed + subject)
        idx = np.arange(len(x_train))
        rng.shuffle(idx)
        split = int(len(idx) * 0.9)
        tr_idx, va_idx = idx[:split], idx[split:]

        model, hist = train_classifier(
            x_train[tr_idx], y_train[tr_idx],
            x_train[va_idx], y_train[va_idx],
            cfg,
            use_cbam=True,
            use_encoder=True,
            bidirectional=True,
            sequence_model="lstm",
        )

        probs = model.predict(x_test, verbose=0)
        preds = probs.argmax(axis=1)
        metrics = summarize_metrics(y_test, preds)
        metrics["subject"] = int(subject)
        rows.append(metrics)

        fold_preds.append({
            "subject": int(subject),
            "y_true": y_test.tolist(),
            "y_pred": preds.tolist(),
        })
        print(f"Finished subject {subject:02d} | acc={metrics['accuracy']:.4f}")

    df, summary = aggregate_fold_metrics(rows)
    return df, summary, fold_preds

In [ ]:
# Set cfg.loso_subjects_limit = None for the full 48-subject run.
# For quick testing, set a small number such as 3 or 5.
cfg.loso_subjects_limit = 3
full_df, full_summary, full_fold_preds = run_loso_full_model(cfg)
full_df.head(), full_summary

In [ ]:
save_json(full_summary, cfg.output_root / "metrics" / "full_model_summary.json")
full_df.to_csv(cfg.output_root / "metrics" / "full_model_folds.csv", index=False)

## Baselines

In [ ]:
def run_loso_psd_svm(cfg):
    rows = []
    subjects = manifest["subject"].tolist()
    if cfg.loso_subjects_limit is not None:
        subjects = subjects[:cfg.loso_subjects_limit]

    for subject in subjects:
        train_rows = manifest[manifest["subject"] != subject]
        test_rows = manifest[manifest["subject"] == subject]
        x_train, y_train, _ = build_dataset_for_subjects(train_rows, cfg)
        x_test, y_test, _ = build_dataset_for_subjects(test_rows, cfg)
        preds = psd_svm_baseline(x_train, y_train, x_test, seed=cfg.seed)
        metrics = summarize_metrics(y_test, preds)
        metrics["subject"] = int(subject)
        rows.append(metrics)
    return aggregate_fold_metrics(rows)

psd_df, psd_summary = run_loso_psd_svm(cfg)
psd_summary

In [ ]:
def run_single_variant(cfg, use_cbam=True, use_encoder=True, bidirectional=True, sequence_model="lstm"):
    rows = []
    subjects = manifest["subject"].tolist()
    if cfg.loso_subjects_limit is not None:
        subjects = subjects[:cfg.loso_subjects_limit]

    for subject in subjects:
        train_rows = manifest[manifest["subject"] != subject]
        test_rows = manifest[manifest["subject"] == subject]
        x_train, y_train, _ = build_dataset_for_subjects(train_rows, cfg)
        x_test, y_test, _ = build_dataset_for_subjects(test_rows, cfg)

        rng = np.random.default_rng(cfg.seed + subject)
        idx = np.arange(len(x_train))
        rng.shuffle(idx)
        split = int(len(idx) * 0.9)
        tr_idx, va_idx = idx[:split], idx[split:]

        model, _ = train_classifier(
            x_train[tr_idx], y_train[tr_idx],
            x_train[va_idx], y_train[va_idx],
            cfg,
            use_cbam=use_cbam,
            use_encoder=use_encoder,
            bidirectional=bidirectional,
            sequence_model=sequence_model,
        )

        preds = model.predict(x_test, verbose=0).argmax(axis=1)
        metrics = summarize_metrics(y_test, preds)
        metrics["subject"] = int(subject)
        rows.append(metrics)
    return aggregate_fold_metrics(rows)

In [ ]:
cnn_df, cnn_summary = run_single_variant(cfg, use_cbam=False, use_encoder=False, bidirectional=False, sequence_model="cnn")
bilstm_lstm_df, bilstm_lstm_summary = run_single_variant(cfg, use_cbam=False, use_encoder=True, bidirectional=True, sequence_model="lstm")
pd.DataFrame([
    {"model": "Full VAE-CBAM-BiLSTM", **full_summary},
    {"model": "PSD-SVM", **psd_summary},
    {"model": "CNN", **cnn_summary},
    {"model": "BLSTM-LSTM", **bilstm_lstm_summary},
]).round(4)

## Ablation study

In [ ]:
ablation_rows = []
variants = [
    ("Full", dict(use_cbam=True, use_encoder=True, bidirectional=True, sequence_model="lstm")),
    ("No_VAE", dict(use_cbam=True, use_encoder=False, bidirectional=True, sequence_model="lstm")),
    ("No_CBAM", dict(use_cbam=False, use_encoder=True, bidirectional=True, sequence_model="lstm")),
    ("UniLSTM", dict(use_cbam=True, use_encoder=True, bidirectional=False, sequence_model="lstm")),
    ("CNN_only", dict(use_cbam=True, use_encoder=True, bidirectional=False, sequence_model="cnn")),
]
for name, kwargs in variants:
    df_v, summary_v = run_single_variant(cfg, **kwargs)
    ablation_rows.append({"variant": name, **summary_v})
pd.DataFrame(ablation_rows).round(4)

## Window-size sensitivity

In [ ]:
def run_window_sensitivity(window_seconds_list=(5, 10, 15), cfg=None):
    out = []
    for win in window_seconds_list:
        cfg_local = cfg
        rows = []
        subjects = manifest["subject"].tolist()
        if cfg_local.loso_subjects_limit is not None:
            subjects = subjects[:cfg_local.loso_subjects_limit]
        for subject in subjects:
            train_rows = manifest[manifest["subject"] != subject]
            test_rows = manifest[manifest["subject"] == subject]
            x_train, y_train, _ = build_dataset_for_subjects(train_rows, cfg_local, window_seconds=win, sequence_length=int(win / cfg_local.frame_hop_seconds))
            x_test, y_test, _ = build_dataset_for_subjects(test_rows, cfg_local, window_seconds=win, sequence_length=int(win / cfg_local.frame_hop_seconds))
            rng = np.random.default_rng(cfg_local.seed + subject)
            idx = np.arange(len(x_train))
            rng.shuffle(idx)
            split = int(len(idx) * 0.9)
            tr_idx, va_idx = idx[:split], idx[split:]
            model, _ = train_classifier(
                x_train[tr_idx], y_train[tr_idx],
                x_train[va_idx], y_train[va_idx],
                cfg_local,
                use_cbam=True, use_encoder=True, bidirectional=True, sequence_model="lstm"
            )
            preds = model.predict(x_test, verbose=0).argmax(axis=1)
            metrics = summarize_metrics(y_test, preds)
            rows.append(metrics)
        _, summary = aggregate_fold_metrics(rows)
        out.append({"window_seconds": win, **summary})
    return pd.DataFrame(out)

window_df = run_window_sensitivity(cfg=cfg)
window_df.round(4)

## Temporal-length sensitivity

In [ ]:
length_df = run_window_sensitivity(window_seconds_list=(5, 10, 15), cfg=cfg)
length_df = length_df.rename(columns={"window_seconds": "sequence_length_frames"})
length_df.round(4)

## Statistical significance

This compares the full model against selected baselines across subject folds using a one-sided Wilcoxon signed-rank test.

In [ ]:
sig_vs_psd = wilcoxon_vs_full(full_df["accuracy"].tolist(), psd_df["accuracy"].tolist())
sig_vs_cnn = wilcoxon_vs_full(full_df["accuracy"].tolist(), cnn_df["accuracy"].tolist())
sig_vs_bilstm = wilcoxon_vs_full(full_df["accuracy"].tolist(), bilstm_lstm_df["accuracy"].tolist())
pd.DataFrame([
    {"comparison": "Full vs PSD-SVM", **sig_vs_psd},
    {"comparison": "Full vs CNN", **sig_vs_cnn},
    {"comparison": "Full vs BLSTM-LSTM", **sig_vs_bilstm},
]).round(6)

## Confusion matrix and classification report

In [ ]:
# rebuild aggregate y_true / y_pred from the full-model folds
y_true_all = []
y_pred_all = []
for fold in full_fold_preds:
    y_true_all.extend(fold["y_true"])
    y_pred_all.extend(fold["y_pred"])

report_df, cm = classification_tables(y_true_all, y_pred_all, CLASS_NAMES)
report_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Grad-CAM example

In [ ]:
# Train one model for a Grad-CAM demo using the first LOSO fold in the current subject limit
subject = manifest["subject"].tolist()[0]
train_rows = manifest[manifest["subject"] != subject]
test_rows = manifest[manifest["subject"] == subject]
x_train, y_train, _ = build_dataset_for_subjects(train_rows, cfg)
x_test, y_test, _ = build_dataset_for_subjects(test_rows, cfg)

rng = np.random.default_rng(cfg.seed + subject)
idx = np.arange(len(x_train))
rng.shuffle(idx)
split = int(len(idx) * 0.9)
tr_idx, va_idx = idx[:split], idx[split:]

grad_model, _ = train_classifier(
    x_train[tr_idx], y_train[tr_idx],
    x_train[va_idx], y_train[va_idx],
    cfg,
    use_cbam=True, use_encoder=True, bidirectional=True, sequence_model="lstm",
)

sample = x_test[:1]
pred = grad_model.predict(sample, verbose=0).argmax(axis=1)[0]
heatmap = make_gradcam_heatmap(grad_model, sample, class_index=int(pred))
pred, CLASS_NAMES[pred], heatmap.shape

In [ ]:
plt.figure(figsize=(5,4))
plt.imshow(heatmap)
plt.title(f"Grad-CAM heatmap | predicted={CLASS_NAMES[pred]}")
plt.axis("off")
plt.show()

## Save final artifacts

In [ ]:
summary_table = pd.DataFrame([
    {"model": "Full VAE-CBAM-BiLSTM", **full_summary},
    {"model": "PSD-SVM", **psd_summary},
    {"model": "CNN", **cnn_summary},
    {"model": "BLSTM-LSTM", **bilstm_lstm_summary},
]).round(4)

summary_table.to_csv(cfg.output_root / "metrics" / "summary_table.csv", index=False)
window_df.to_csv(cfg.output_root / "metrics" / "window_sensitivity.csv", index=False)
length_df.to_csv(cfg.output_root / "metrics" / "temporal_length_sensitivity.csv", index=False)
report_df.to_csv(cfg.output_root / "metrics" / "classification_report.csv")
np.save(cfg.output_root / "metrics" / "confusion_matrix.npy", cm)
summary_table